In [1]:
# Mount Google Drive
from google.colab import drive
#drive.mount('/content/drive',force_remount=True)
drive.mount('/content/drive')

Mounted at /content/drive


**LightGBM and Statistical Benchmark models for Vallés-Pérez et al. Seq2Seq paper (2022)**



**Reference** :
Vallés-Pérez et al. (2022). Approaching sales forecasting using recurrent
neural networks and transformers. *Expert Systems with Applications*, 201, 117249.
https://doi.org/10.1016/j.eswa.2022.117249

* ***Dataset:** Corporación Favorita (Kaggle), daily granularity

* **Series:** 221,184 (4,096 items x 54 stores, full cartesian product, zero-filled) —
as constructed by the paper's own pipeline (github.com/ivallesp/cFavorita)

* **Train:** January 2013 – June 28 2017

* **Test:** Period 1 — June 29 to July 14 2017 (h=16 daily steps, paper Table 1)

* **LGBM model:** Direct multi-step. All test features fixed at the training cutoff
; a single model predicts all 16 test days from that state.
Loss is weighted RMSE on the log1p target, which equals the paper's RMSWLE.
seed=655321 (the paper's first run seed).

* **Statistical:** ADI-based demand classification — AutoETS(best model is chosen based on least AICc by this model itself) for
smooth demand (ADI<1.32), CrostonSBA for non-smooth (ADI>=1.32). Mean forecast is the fallback if min_obs criterion is not satisfied

* **Prerequisites:** Checkpoint files generated by the cFavorita pipeline. See
'prepare_data_case1.py' in the repository, or reuse checkpoints from the
seq2seq reproduction run. Required in 'CKPT_PATH':

    * sales.npy       log1p unit_sales,      shape (221184, n_days)
    * weights.npy     perishable weights,    shape (221184,)
    * config.csv      cutoff, horizon, test period dates
    * num_<feat>.npy  numeric time features, shape (221184, n_days) each
    * cat_<feat>.npy  static item/store features, shape (221184,) each

* Required packages: lightgbm, statsforecast, pandas, numpy

**Import libraries**

In [2]:
#install libraries if not available
!pip install lightgbm statsforecast pandas numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 501.0/501.0 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 348.6/348.6 kB 31.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 281.0/281.0 kB 30.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.6/46.6 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.9/59.9 kB 9.1 MB/s eta 0:00:00


In [3]:
## import required packages

import os
import gc
import time
import logging
import warnings
import numpy as np
import pandas as pd
import lightgbm as lgb
from statsforecast import StatsForecast
from statsforecast.models import AutoETS, CrostonSBA

# suppress warnings and enable logging
warnings.filterwarnings("ignore")

**Set paths and config**

In [4]:
##### paths #######################################
OUTPUT_PATH = "/content/drive/MyDrive/tll_reproducibility/seq2seq/daily_benchmark"   # replace with your path
BASE_DIR = "/content/drive/MyDrive/seq2seq_results/daily_benchmark"
CKPT_PATH_DATA   = os.path.join(BASE_DIR, "checkpoints")
CKPT_PATH_RESULTS = os.path.join(OUTPUT_PATH, "checkpoints")
RESULTS_DIR = OUTPUT_PATH
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(CKPT_PATH_RESULTS, exist_ok=True)

# logging — file handler plus console. force=True is required in Colab
LOG_PATH = os.path.join(RESULTS_DIR, "run_log.txt")
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(message)s",
    datefmt="%H:%M:%S",
    handlers=[logging.FileHandler(LOG_PATH, mode="a"), logging.StreamHandler()],
    force=True,
)
logger = logging.getLogger(__name__)
logger.info(f"Logging to {LOG_PATH}")


########## config #######
ADI_THRESH   = 1.32          # Syntetos & Boylan (2005)
SEED         = 655321        # paper's first run seed
LAGS         = [1, 2, 3, 4, 5, 6, 7, 14, 21, 28]
ROLLING_WINS = [7, 14, 28]
BATCH_SERIES = 1000          # series per batch for feature building — lower if OOM
SF_BATCH     = 20000         # series per batch for statsforecast fitting
MIN_OBS      = 2 # min daily positive demand observations for statsforecast fitting as a crash guard

# features known in advance at forecast time (use the actual test-day value)
FORWARD_FEATS = {
    "onpromotion", "holidays_transferred", "holidays_count",
    "dcoilwtico", "year", "month", "day", "dayofweek",
}
# features not available for the test period (use last training day value)
HISTORICAL_FEATS = {"transactions"}

np.random.seed(SEED)

# LGBM params — matching the seq2seq loss function (weighted RMSE on log1p = RMSWLE)
LGBM_PARAMS = {
    "objective"        : "regression",
    "metric"           : "rmse",
    "learning_rate"    : 0.005,
    "num_leaves"       : 63,
    "max_depth"        : 6,
    "min_child_samples": 20,
    "subsample"        : 0.8,
    "colsample_bytree" : 0.8,
    "verbose"          : -1,
    "seed"             : SEED,
    "n_jobs"           : -1,
}
NUM_ROUNDS     = 3000
EARLY_STOPPING = 100

09:12:51  Logging to /content/drive/MyDrive/tll_reproducibility/seq2seq/daily_benchmark/run_log.txt


**Import data**

In [5]:
##### load data ##################

logger.info("Loading checkpoint files...")
t0 = time.time()

sales   = np.load(os.path.join(CKPT_PATH_DATA, "sales.npy"))      # log1p unit_sales
weights = np.load(os.path.join(CKPT_PATH_DATA, "weights.npy"))    # perishable weights
config  = pd.read_csv(os.path.join(CKPT_PATH_DATA, "config.csv")).iloc[0]

n_series         = int(config["n_series"])
n_days           = int(config["n_days"])
cutoff           = int(config["cutoff"])
FORECAST_HORIZON = int(config["forecast_horizon"])

logger.info(f"  sales.npy   : {sales.shape}  ({time.time()-t0:.1f}s)")
logger.info(f"  Series      : {n_series:,}  Days: {n_days}  Cutoff: {cutoff}")
logger.info(f"  Period 1    : {config['period1_start']} -> {config['period1_end']}  h={FORECAST_HORIZON}")

# time-varying numeric features, shape (n_series, n_days) each
num_files  = sorted([f for f in os.listdir(CKPT_PATH_DATA)
                     if f.startswith("num_") and f.endswith(".npy")])
num_arrays = {f[4:-4]: np.load(os.path.join(CKPT_PATH_DATA, f)) for f in num_files}

# static item/store features, shape (n_series,) each
cat_files  = sorted([f for f in os.listdir(CKPT_PATH_DATA)
                     if f.startswith("cat_") and f.endswith(".npy")])
cat_arrays = {f[4:-4]: np.load(os.path.join(CKPT_PATH_DATA, f)) for f in cat_files}

logger.info(f"  Numeric feats ({len(num_arrays)}): {list(num_arrays.keys())}")
logger.info(f"  Static feats  ({len(cat_arrays)}): {list(cat_arrays.keys())}")

# every numeric feature must be classified as forward-looking or historical
for feat in num_arrays:
    if feat not in FORWARD_FEATS and feat not in HISTORICAL_FEATS:
        raise ValueError(f"Feature '{feat}' not classified in FORWARD_FEATS/HISTORICAL_FEATS")

# guard against a wrong cutoff pointing at the Kaggle zero-filled test window
test_check = sales[:, cutoff:cutoff + FORECAST_HORIZON]
assert (test_check > 0).sum() > 0, "CRITICAL: all test actuals are zero — cutoff is wrong"
logger.info(f"  Test non-zero: {(test_check > 0).sum():,} / {test_check.size:,} "
            f"({100*(test_check>0).mean():.1f}%)")

09:12:55  Loading checkpoint files...
09:13:41    sales.npy   : (221184, 1700)  (46.5s)
09:13:41    Series      : 221,184  Days: 1700  Cutoff: 1636
09:13:41    Period 1    : 2017-06-29 -> 2017-07-14  h=16
09:18:57    Numeric feats (9): ['day', 'dayofweek', 'dcoilwtico', 'holidays_count', 'holidays_transferred', 'month', 'onpromotion', 'transactions', 'year']
09:18:57    Static feats  (11): ['holidays_locale', 'holidays_locale_name', 'holidays_type', 'item_family', 'item_nbr', 'item_perishable', 'store_city', 'store_cluster', 'store_nbr', 'store_state', 'store_type']
09:18:57    Test non-zero: 1,699,183 / 3,538,944 (48.0%)


**Statistical Benchmarks: Sales series classification**

In [6]:
###### Sales series (demand proxy) classification and statistical model training data ##################
# ADI-based split: smooth (ADI<1.32) -> AutoETS, non-smooth (ADI>=1.32) -> CrostonSBA
# Classification is done on ORIGINAL unit sales, using training days only.
# Each column of the sales matrix is one calendar day (the cFavorita pipeline
# zero-fills every date), so ADI = n_train_days / n_nonzero_days.

sbc_path          = os.path.join(CKPT_PATH_RESULTS, "sbc_log.csv")
long_path         = os.path.join(CKPT_PATH_RESULTS, "df_long.csv")
test_actuals_path = os.path.join(CKPT_PATH_RESULTS, "test_actuals.csv")

#### Step 1: classification
logger.info("Series classification (smooth/non-smooth) on training period...")
t0 = time.time()

sbc_rows = []
for batch_start in range(0, n_series, BATCH_SERIES):
    batch_end  = min(batch_start + BATCH_SERIES, n_series)
    batch_orig = np.expm1(np.clip(sales[batch_start:batch_end, :cutoff], 0, None))

    for i in range(batch_orig.shape[0]):
        demand = batch_orig[i].astype(np.float64)
        n_nz   = int((demand > 0).sum())

        if n_nz == 0:
            cat = "non-smooth"                     # ADI = inf
        else:
            adi = len(demand) / n_nz
            cat = "smooth" if adi < ADI_THRESH else "non-smooth"

        sbc_rows.append({"unique_id": str(batch_start + i),
                         "category" : cat,
                         "weight"   : float(weights[batch_start + i])})

    del batch_orig
    if batch_end % 40000 == 0 or batch_end == n_series:
        logger.info(f"  {batch_end:,}/{n_series:,} series  [{(time.time()-t0)/60:.1f} min]")

df_sbc = pd.DataFrame(sbc_rows)
df_sbc.to_csv(sbc_path, index=False)          # overwrites any earlier version
del sbc_rows
gc.collect()

logger.info(f"  Classification done in {(time.time()-t0)/60:.1f} min")
logger.info(f"  Distribution:\n{df_sbc['category'].value_counts().to_string()}")

#### Step 2: df_long for statsforecast — cacheable###
# Content is the training sales matrix in original units, independent of
# how series are classified, so an existing file can be reused.
if os.path.exists(long_path):
    logger.info("  Cached df_long.csv found — reusing (independent of classification)")
else:
    logger.info("  Building df_long.csv (original units, training days only)...")
    t0 = time.time()
    first_write = True

    for batch_start in range(0, n_series, BATCH_SERIES):
        batch_end  = min(batch_start + BATCH_SERIES, n_series)
        batch_orig = np.expm1(np.clip(sales[batch_start:batch_end, :cutoff], 0, None))

        n_b, n_t  = batch_orig.shape
        long_rows = pd.DataFrame({
            "unique_id": np.repeat([str(batch_start + i) for i in range(n_b)], n_t),
            "ds"       : np.tile(np.arange(n_t), n_b),
            "y"        : np.clip(batch_orig.reshape(-1), 0, None).astype(np.float64),
        })
        long_rows.to_csv(long_path, mode="w" if first_write else "a",
                         header=first_write, index=False)
        first_write = False

        del long_rows, batch_orig
        gc.collect()

        if batch_end % 20000 == 0 or batch_end == n_series:
            logger.info(f"  {batch_end:,}/{n_series:,} series  [{(time.time()-t0)/60:.1f} min]")

    logger.info(f"  df_long.csv written in {(time.time()-t0)/60:.1f} min")

#### Step 3: test actuals in original units — cacheable
if os.path.exists(test_actuals_path):
    logger.info("  Cached test_actuals.csv found — reusing")
else:
    test_orig = np.expm1(np.clip(sales[:, cutoff:cutoff + FORECAST_HORIZON], 0, None))
    df_test = pd.DataFrame({
        "unique_id": np.repeat([str(i) for i in range(n_series)], FORECAST_HORIZON),
        "ds"       : np.tile(np.arange(cutoff, cutoff + FORECAST_HORIZON), n_series),
        "actual"   : test_orig.reshape(-1).astype(np.float64),
        "weight"   : np.repeat(weights, FORECAST_HORIZON).astype(np.float64),
    })
    df_test.to_csv(test_actuals_path, index=False)
    del test_orig, df_test
    gc.collect()
    logger.info("  test_actuals.csv written")

09:18:57  Series classification (smooth/non-smooth) on training period...
09:18:57    40,000/221,184 series  [0.0 min]
09:18:57    80,000/221,184 series  [0.0 min]
09:18:58    120,000/221,184 series  [0.0 min]
09:18:58    160,000/221,184 series  [0.0 min]
09:18:58    200,000/221,184 series  [0.0 min]
09:18:58    221,184/221,184 series  [0.0 min]
09:18:59    Classification done in 0.0 min
09:18:59    Distribution:
category
non-smooth    185953
smooth         35231
09:18:59    Cached df_long.csv found — reusing (independent of classification)
09:18:59    Cached test_actuals.csv found — reusing


In [7]:
df_sbc.category.value_counts()

,count
category,
non-smooth,185953
smooth,35231


In [8]:
185953/(185953+35231)

0.8407163266782407

In [11]:
# df_long = pd.read_csv(long_path, dtype={"unique_id": str}) #just checking median length of training series
# smooth_ids     = df_sbc.loc[df_sbc.category == "smooth", "unique_id"].tolist()
# n_days_per_series = df_long[df_long.unique_id.isin(smooth_ids)].groupby("unique_id")["y"].count()
# print(n_days_per_series.describe())

**Statistical Benchmark: AutoETS for smooth and SBA for non-smooth categories**

In [13]:
######## statistical models #######################################
# AutoETS(season_length=7) performs an internal model search across error,
# trend, and seasonal components and selects the specification minimizing
# its own AICc. It does not force seasonality just because season_length>1,
# and returns a non-seasonal ETS specification where seasonality doesn't
# improve the fit. CrostonSBA is the Syntetos-Boylan bias-corrected Croston variant.

logger.info("Fitting statistical models...")
t0 = time.time()

stat_pred_path = os.path.join(RESULTS_DIR, "statistical_predictions_1.csv")

def is_seasonal(method_str):
    """ETS(E,T,S) notation"""
    return method_str.split(",")[-1].rstrip(")") != "N"

if os.path.exists(stat_pred_path):
    logger.info("  Cached statistical_predictions.csv found — loading")
    df_stat_fc = pd.read_csv(stat_pred_path, dtype={"unique_id": str})
else:
    logger.info("  Loading df_long.csv (original units)...")
    df_long = pd.read_csv(long_path, dtype={"unique_id": str})
    df_long["ds"] = df_long["ds"].astype(int)
    df_long["y"]  = df_long["y"].clip(lower=0).astype(float)

    # scale guard — df_long must hold original units, not log1p
    assert df_long["y"].max() > 10, "df_long looks like log1p"
    logger.info(f"  df_long: {df_long.shape}  max y={df_long['y'].max():.1f}")

    def fit_stat(df_long, ids, model, label, h, min_obs=2):
        """Fit one statsforecast model over a set of series, batched to limit RAM.
        Series with < min_obs days of actual demand fall back to their mean."""
        if not ids:
            return pd.DataFrame(columns=["unique_id", "ds", "pred"])
        df_sub    = df_long[df_long.unique_id.isin(ids)]
        n_obs     = df_sub.groupby("unique_id")["y"].count()
        valid_ids = n_obs[n_obs >= min_obs].index.tolist()
        tiny_ids  = n_obs[n_obs <  min_obs].index.tolist()
        logger.info(f"{label}: {len(valid_ids):,} fitted  {len(tiny_ids):,} mean fallback")

        results = []
        for i in range(0, len(valid_ids), SF_BATCH):
            chunk  = valid_ids[i:i + SF_BATCH]
            df_chunk = df_sub[df_sub.unique_id.isin(chunk)]
            sf  = StatsForecast(models=[model], freq=1, n_jobs=-1)
            fc  = sf.forecast(df=df_chunk, h=h).reset_index(drop=True)
            col = [c for c in fc.columns if c not in ["unique_id", "ds"]][0]
            fc  = fc.rename(columns={col: "pred"})[["unique_id", "ds", "pred"]]
            fc["pred"] = fc["pred"].clip(lower=0)
            results.append(fc)
            logger.info(f"{min(i+SF_BATCH, len(valid_ids)):,}/{len(valid_ids):,} "
                        f"[{(time.time()-t0)/60:.1f} min]")
            del df_chunk
            gc.collect()

        if tiny_ids:
            means = df_sub[df_sub.unique_id.isin(tiny_ids)].groupby("unique_id")["y"].mean()
            rows  = [{"unique_id": uid, "ds": int(n_obs.get(uid, 0)) + d,
                      "pred": max(0.0, float(means.get(uid, 0.0)))}
                    for uid in tiny_ids for d in range(h)]
            results.append(pd.DataFrame(rows))

        return pd.concat(results, ignore_index=True) if results else \
               pd.DataFrame(columns=["unique_id", "ds", "pred"])

    smooth_ids     = df_sbc.loc[df_sbc.category == "smooth", "unique_id"].tolist()
    non_smooth_ids = df_sbc.loc[df_sbc.category == "non-smooth", "unique_id"].tolist()

    sf_ets = StatsForecast(models=[AutoETS(season_length=7)], freq=1, n_jobs=-1)
    sf_ets.fit(df=df_long[df_long.unique_id.isin(smooth_ids)])
    methods = {uid: sf_ets.fitted_[i, 0].model_["method"] for i, uid in enumerate(sf_ets.uids)}
    selected_model = pd.Series(methods).apply(lambda m: "AutoETS_seas" if is_seasonal(m) else "AutoETS_ns")
    logger.info(f"  AutoETS(s=7): {(selected_model=='AutoETS_seas').sum():,} seasonal, "
                f"{(selected_model=='AutoETS_ns').sum():,} non-seasonal")

    fc_ets = fit_stat(df_long, smooth_ids,     AutoETS(season_length=7), "AutoETS(s=7)", FORECAST_HORIZON, min_obs=MIN_OBS)
    fc_sba = fit_stat(df_long, non_smooth_ids, CrostonSBA(), "CrostonSBA",   FORECAST_HORIZON, min_obs=MIN_OBS)

    df_stat_fc = pd.concat([fc_ets, fc_sba], ignore_index=True)
    df_stat_fc.to_csv(stat_pred_path, index=False)

    # AutoETS models chosen for each series
    pd.DataFrame({"unique_id": methods.keys(), "method": methods.values(),
                  "selected_model": selected_model.values}).to_csv(
        os.path.join(RESULTS_DIR, "autoets_model_selection.csv"), index=False)

    del df_long, fc_ets, fc_sba
    gc.collect()

logger.info(f"  Statistical models done in {(time.time()-t0)/60:.1f} min")

10:39:37  Fitting statistical models...
10:39:37    Loading df_long.csv (original units)...
10:40:18    df_long: (361857024, 3)  max y=89440.0
10:45:08    AutoETS(s=7): 29,749 seasonal, 5,482 non-seasonal
10:45:15  AutoETS(s=7): 35,231 fitted  0 mean fallback
10:47:57  20,000/35,231 [8.3 min]
10:50:01  35,231/35,231 [10.4 min]
10:50:19  CrostonSBA: 185,953 fitted  0 mean fallback
10:50:33  20,000/185,953 [10.9 min]
10:50:46  40,000/185,953 [11.1 min]
10:51:00  60,000/185,953 [11.4 min]
10:51:14  80,000/185,953 [11.6 min]
10:51:27  100,000/185,953 [11.8 min]
10:51:41  120,000/185,953 [12.1 min]
10:51:54  140,000/185,953 [12.3 min]
10:52:08  160,000/185,953 [12.5 min]
10:52:21  180,000/185,953 [12.7 min]
10:52:33  185,953/185,953 [12.9 min]
10:52:37    Statistical models done in 13.0 min


In [8]:
# n_days_per_series = df_long[df_long.unique_id.isin(smooth_ids)].groupby("unique_id")["y"].count()
# print(n_days_per_series.describe())

**Machine Learning Benchmark: LGBM feature build**

In [14]:
##### LGBM feature builders — no leakage #####################

n_lags  = len(LAGS)
n_roll  = len(ROLLING_WINS) * 2
n_num   = len(num_arrays)
n_cat   = len(cat_arrays)
n_feats = n_lags + n_roll + n_num + n_cat

start_idx  = max(max(LAGS), max(ROLLING_WINS))   # first day with full lag history
train_rows = cutoff - start_idx
val_days   = FORECAST_HORIZON

logger.info(f"Features: {n_feats} (lags={n_lags}, rolling={n_roll}, "
            f"numeric={n_num}, categorical={n_cat})")
logger.info(f"start_idx={start_idx}  train_rows={train_rows}  val_days={val_days}")


def lag_features_vec(arr, lags):
    """lag_l at day d = sales[d-l]. Zero-padded at the start. No leakage."""
    n_s, n_d = arr.shape
    out = np.zeros((n_s, n_d, len(lags)), dtype=np.float32)
    for i, lag in enumerate(lags):
        if lag < n_d:
            out[:, lag:, i] = arr[:, :n_d - lag]
    return out


def rolling_mean_vec(arr, window):
    """Rolling mean at day d = mean(sales[max(0,d-w):d]) — excludes sales[d]."""
    n_s, n_d = arr.shape
    cs = np.concatenate([np.zeros((n_s, 1), dtype=np.float32),
                         np.cumsum(arr, axis=1).astype(np.float32)], axis=1)
    out = np.zeros((n_s, n_d), dtype=np.float32)
    out[:, window:] = (cs[:, window:-1] - cs[:, :n_d - window]) / window
    for d in range(1, window):
        out[:, d] = cs[:, d] / d
    return out


def rolling_std_vec(arr, window):
    """Rolling std at day d = std(sales[max(0,d-w):d]) — excludes sales[d]."""
    n_s, n_d = arr.shape
    cs    = np.concatenate([np.zeros((n_s, 1), dtype=np.float32),
                            np.cumsum(arr, axis=1).astype(np.float32)], axis=1)
    cs_sq = np.concatenate([np.zeros((n_s, 1), dtype=np.float32),
                            np.cumsum(arr ** 2, axis=1).astype(np.float32)], axis=1)
    out = np.zeros((n_s, n_d), dtype=np.float32)
    mean_sq = (cs_sq[:, window:-1] - cs_sq[:, :n_d - window]) / window
    mean    = (cs[:,    window:-1] - cs[:,    :n_d - window]) / window
    out[:, window:] = np.sqrt(np.maximum(mean_sq - mean ** 2, 0))
    for d in range(2, window):
        mean_d = cs[:, d] / d
        var_d  = cs_sq[:, d] / d - mean_d ** 2
        out[:, d] = np.sqrt(np.maximum(var_d, 0))
    return out


def build_features_for_batch(s_sales, s_num, s_cat, days_slice):
    """Feature matrix for one batch of series over a slice of days."""
    n_batch   = s_sales.shape[0]
    n_d_slice = len(range(*days_slice.indices(s_sales.shape[1])))

    lag_stack = lag_features_vec(s_sales, LAGS)[:, days_slice, :]

    roll_feats = []
    for win in ROLLING_WINS:
        roll_feats.append(rolling_mean_vec(s_sales, win)[:, days_slice])
        roll_feats.append(rolling_std_vec(s_sales,  win)[:, days_slice])
    roll_stack = np.stack(roll_feats, axis=2)

    num_stack = np.stack([arr[:, days_slice] for arr in s_num.values()],
                         axis=2).astype(np.float32)

    cat_mat   = np.stack(list(s_cat.values()), axis=1).astype(np.float32)
    cat_tiled = np.broadcast_to(cat_mat[:, np.newaxis, :],
                                (n_batch, n_d_slice, n_cat)).copy()

    feats = np.concatenate([lag_stack, roll_stack, num_stack, cat_tiled],
                           axis=2).astype(np.float32)
    assert feats.shape == (n_batch, n_d_slice, n_feats), f"Shape mismatch: {feats.shape}"
    return feats.reshape(-1, n_feats)


# leakage unit tests — rolling windows must exclude the current day
_t  = np.array([[1., 2., 3., 4., 5.]], dtype=np.float32)
_rm = rolling_mean_vec(_t, 3)
assert abs(_rm[0, 3] - 2.0) < 1e-5, f"Rolling mean leakage at d=3: {_rm[0,3]}"
assert abs(_rm[0, 4] - 3.0) < 1e-5, f"Rolling mean leakage at d=4: {_rm[0,4]}"
_rs = rolling_std_vec(_t, 3)
assert abs(_rs[0, 3] - np.std([1., 2., 3.])) < 1e-4, "Rolling std leakage at d=3"
logger.info("  Rolling leakage unit tests passed")

10:53:44  Features: 36 (lags=10, rolling=6, numeric=9, categorical=11)
10:53:44  start_idx=28  train_rows=1608  val_days=16
10:53:44    Rolling leakage unit tests passed



**Machine Learning Benchmark: LGBM direct multi-step forecasting**


In [15]:
USE_MEMMAP = False # True keeps X_tr on disk — needed if RAM < ~60 GB

In [16]:
##### LGBM training ##################

lgbm_pred_path = os.path.join(CKPT_PATH_RESULTS, "lgbm_preds.npy")

if os.path.exists(lgbm_pred_path):
    logger.info("Cached lgbm_preds.npy found — loading")
    lgbm_preds = np.load(lgbm_pred_path)
    y_test     = np.load(os.path.join(CKPT_PATH_RESULTS, "y_test.npy"))
    w_test     = np.load(os.path.join(CKPT_PATH_RESULTS, "w_test.npy"))
else:
    #### training feature matrices, built in batches ####
    # X_tr and X_val are pre-allocated and filled directly inside the batch
    # loop. Building one combined matrix and then splitting it with a boolean
    # mask allocates two extra full copies of a ~51 GB array. Row order and
    # values here are identical to that approach.
    logger.info(f"Building training features in batches of {BATCH_SERIES:,}...")
    t0 = time.time()
    train_slice = slice(start_idx, cutoff)

    train_days_tr = train_rows - val_days      # days per series used for fitting
    n_tr_rows     = n_series * train_days_tr
    n_val_rows    = n_series * val_days

    logger.info(f"  X_tr  : ({n_tr_rows:,}, {n_feats}) "
                f"= {n_tr_rows * n_feats * 4 / 1e9:.1f} GB")
    logger.info(f"  X_val : ({n_val_rows:,}, {n_feats}) "
                f"= {n_val_rows * n_feats * 4 / 1e9:.1f} GB")

    if USE_MEMMAP:
        xtr_path = os.path.join(CKPT_PATH_RESULTS, "X_tr.memmap")
        X_tr = np.memmap(xtr_path, dtype=np.float32, mode="w+",
                         shape=(n_tr_rows, n_feats))
        logger.info(f"  X_tr backed by disk: {xtr_path}")
    else:
        xtr_path = None
        X_tr = np.zeros((n_tr_rows, n_feats), dtype=np.float32)

    y_tr  = np.zeros(n_tr_rows,  dtype=np.float32)
    w_tr  = np.zeros(n_tr_rows,  dtype=np.float32)
    X_val = np.zeros((n_val_rows, n_feats), dtype=np.float32)
    y_val = np.zeros(n_val_rows, dtype=np.float32)
    w_val = np.zeros(n_val_rows, dtype=np.float32)

    row_tr, row_val = 0, 0

    for batch_start in range(0, n_series, BATCH_SERIES):
        batch_end = min(batch_start + BATCH_SERIES, n_series)
        n_batch   = batch_end - batch_start

        s_sales = sales[batch_start:batch_end]
        s_num   = {f: a[batch_start:batch_end] for f, a in num_arrays.items()}
        s_cat   = {f: a[batch_start:batch_end] for f, a in cat_arrays.items()}

        Xb  = build_features_for_batch(s_sales, s_num, s_cat, train_slice)
        Xb3 = Xb.reshape(n_batch, train_rows, n_feats)
        yb  = s_sales[:, start_idx:cutoff].astype(np.float32)      # (n_batch, train_rows)
        wb  = weights[batch_start:batch_end].astype(np.float32)    # (n_batch,)

        # time-based split by day: the last val_days of each series go to
        # validation — equivalent to mask_2d[:, -val_days:] = True
        n_add_tr = n_batch * train_days_tr
        X_tr[row_tr:row_tr + n_add_tr] = Xb3[:, :train_days_tr, :].reshape(-1, n_feats)
        y_tr[row_tr:row_tr + n_add_tr] = yb[:, :train_days_tr].reshape(-1)
        w_tr[row_tr:row_tr + n_add_tr] = np.repeat(wb, train_days_tr)
        row_tr += n_add_tr

        n_add_val = n_batch * val_days
        X_val[row_val:row_val + n_add_val] = Xb3[:, train_days_tr:, :].reshape(-1, n_feats)
        y_val[row_val:row_val + n_add_val] = yb[:, train_days_tr:].reshape(-1)
        w_val[row_val:row_val + n_add_val] = np.repeat(wb, val_days)
        row_val += n_add_val

        del Xb, Xb3, yb, s_sales, s_num, s_cat
        gc.collect()

        if batch_end % 20000 == 0 or batch_end == n_series:
            logger.info(f"  {batch_end:,}/{n_series:,}  [{(time.time()-t0)/60:.1f} min]")

    assert row_tr  == n_tr_rows,  f"train row mismatch: {row_tr} vs {n_tr_rows}"
    assert row_val == n_val_rows, f"val row mismatch: {row_val} vs {n_val_rows}"
    if USE_MEMMAP:
        X_tr.flush()

    logger.info(f"  Train: {row_tr:,}  Val: {row_val:,} "
                f"(last {val_days} days/series)  [{(time.time()-t0)/60:.1f} min]")
    logger.info(f"  y_tr non-zero: {(y_tr>0).sum():,} / {len(y_tr):,}")

    #### test feature matrix — direct multi-step, fixed at cutoff ##
    logger.info("Building test features (direct multi-step)...")

    lag_test = np.zeros((n_series, n_lags), dtype=np.float32)
    for i, lag in enumerate(LAGS):
        lag_test[:, i] = sales[:, cutoff - lag]

    roll_test = np.zeros((n_series, n_roll), dtype=np.float32)
    col = 0
    for win in ROLLING_WINS:
        window_data           = sales[:, cutoff - win:cutoff]
        roll_test[:, col]     = window_data.mean(axis=1)
        roll_test[:, col + 1] = window_data.std(axis=1)
        col += 2

    cat_test = np.stack(list(cat_arrays.values()), axis=1).astype(np.float32)

    X_test_3d = np.zeros((n_series, FORECAST_HORIZON, n_feats), dtype=np.float32)
    for h in range(FORECAST_HORIZON):
        test_day = cutoff + h
        X_test_3d[:, h, :n_lags]                = lag_test
        X_test_3d[:, h, n_lags:n_lags + n_roll] = roll_test
        for j, (feat, arr) in enumerate(num_arrays.items()):
            c = n_lags + n_roll + j
            # forward features use the actual test-day value; historical use last training day
            X_test_3d[:, h, c] = arr[:, test_day] if feat in FORWARD_FEATS else arr[:, cutoff - 1]
        X_test_3d[:, h, n_lags + n_roll + n_num:] = cat_test

    X_test = X_test_3d.reshape(-1, n_feats).astype(np.float32)
    del X_test_3d
    gc.collect()

    y_test = sales[:, cutoff:cutoff + FORECAST_HORIZON].reshape(-1).astype(np.float32)
    w_test = np.repeat(weights, FORECAST_HORIZON).astype(np.float32)
    assert len(X_test) == len(y_test) == len(w_test)
    assert (y_test > 0).sum() > 0, "CRITICAL: y_test all zero"

    # lag features must be identical across all 16 horizons (direct multi-step)
    for i, lag in enumerate(LAGS):
        for s in range(min(5, n_series)):
            vals = X_test[s * FORECAST_HORIZON:(s + 1) * FORECAST_HORIZON, i]
            assert np.allclose(vals, vals[0]), f"Lag {lag} not fixed for series {s}"

    # rolling features must match direct computation
    for wi, win in enumerate(ROLLING_WINS):
        direct = sales[:5, cutoff - win:cutoff].mean(axis=1)
        stored = X_test[:5 * FORECAST_HORIZON:FORECAST_HORIZON, n_lags + wi * 2]
        assert np.allclose(direct, stored, atol=1e-5), f"Rolling mean mismatch, window {win}"
    logger.info("  Direct multi-step verification passed")
    logger.info(f"  X_test: {X_test.shape}  y_test non-zero: {(y_test>0).sum():,}")

    np.save(os.path.join(CKPT_PATH_RESULTS, "y_test.npy"), y_test)
    np.save(os.path.join(CKPT_PATH_RESULTS, "w_test.npy"), w_test)

    ######## train ################
    logger.info("Training LightGBM (weighted RMSE on log1p = RMSWLE)...")
    t0 = time.time()

    train_data = lgb.Dataset(X_tr,  label=y_tr,  weight=w_tr)
    val_data   = lgb.Dataset(X_val, label=y_val, weight=w_val, reference=train_data)

    # bin the features now (~1 byte per value instead of 4), then release the
    # float32 matrices before boosting starts
    train_data.construct()
    val_data.construct()
    del X_tr, y_tr, w_tr, X_val, y_val, w_val
    gc.collect()
    logger.info("  Datasets constructed, feature matrices released")

    model_lgbm = lgb.train(
        LGBM_PARAMS, train_data,
        num_boost_round=NUM_ROUNDS,
        valid_sets=[val_data],
        callbacks=[lgb.early_stopping(EARLY_STOPPING, verbose=False),
                   lgb.log_evaluation(period=50)],
    )
    logger.info(f"  Trained in {(time.time()-t0)/60:.1f} min  "
                f"best iteration: {model_lgbm.best_iteration}")

    lgbm_preds = model_lgbm.predict(X_test).astype(np.float32)
    np.save(lgbm_pred_path, lgbm_preds)
    model_lgbm.save_model(os.path.join(CKPT_PATH_RESULTS, "lgbm_model.txt"))

    del train_data, val_data, X_test
    gc.collect()

    if USE_MEMMAP and xtr_path and os.path.exists(xtr_path):
        os.remove(xtr_path)
        logger.info("  X_tr.memmap removed")

10:53:47  Cached lgbm_preds.npy found — loading


In [17]:
import psutil
print(f"RAM: {psutil.virtual_memory().total/1e9:.0f} GB "
      f"available: {psutil.virtual_memory().available/1e9:.0f} GB")

RAM: 190 GB available: 165 GB


**Evaluation**

In [18]:
###### metrics ######################################################
# RMSLE, RMSWLE and MALE are all computed in log1p space, matching the
# paper's Table 1 convention. RMSWLE is the paper's NWRMSLE: perishable
# items carry weight 1.25, non-perishable 1.0.
#
# Reported twice for each model:
#   _all : every test row, including the ~52% where actual sales are zero
#   _nz  : rows where actual sales > 0

def compute_metrics(actual_orig, pred_orig, weights_row):
    """All inputs in ORIGINAL unit sales. Returns log1p-space metrics."""
    a = np.asarray(actual_orig, dtype=np.float64)
    p = np.clip(np.asarray(pred_orig, dtype=np.float64), 0, None)
    w = np.asarray(weights_row,  dtype=np.float64)

    log_err = np.log1p(a) - np.log1p(p)
    return {
        "RMSLE" : round(float(np.sqrt(np.mean(log_err ** 2))), 4),
        "RMSWLE": round(float(np.sqrt(np.average(log_err ** 2, weights=w))), 4),
        "MALE"  : round(float(np.mean(np.abs(log_err))), 4),
    }


def metrics_all_and_nz(actual_orig, pred_orig, weights_row):
    """Compute metrics on all rows and on non-zero-actual rows only."""
    a  = np.asarray(actual_orig, dtype=np.float64)
    m_all = compute_metrics(a, pred_orig, weights_row)
    nz    = a > 0
    m_nz  = compute_metrics(a[nz],
                            np.asarray(pred_orig, dtype=np.float64)[nz],
                            np.asarray(weights_row, dtype=np.float64)[nz])
    return {
        "RMSLE_all" : m_all["RMSLE"],  "RMSWLE_all" : m_all["RMSWLE"],  "MALE_all" : m_all["MALE"],
        "RMSLE_nz"  : m_nz["RMSLE"],   "RMSWLE_nz"  : m_nz["RMSWLE"],   "MALE_nz"  : m_nz["MALE"],
        "n_all"     : int(len(a)),     "n_nz"       : int(nz.sum()),
    }


# ── LGBM: predictions and actuals are in log1p, convert to original ──
y_test_orig  = np.expm1(np.clip(y_test,     0, None))
lgbm_pr_orig = np.expm1(np.clip(lgbm_preds, 0, None))
lgbm_m = metrics_all_and_nz(y_test_orig, lgbm_pr_orig, w_test)
logger.info(f"LGBM  RMSLE_all={lgbm_m['RMSLE_all']}  RMSLE_nz={lgbm_m['RMSLE_nz']}")

# ── Statistical: predictions already in original units ──────────────
df_test = pd.read_csv(os.path.join(CKPT_PATH_RESULTS, "test_actuals.csv"), dtype={"unique_id": str})
df_test["ds"] = df_test["ds"].astype(int)

df_stat_eval = df_test.merge(df_stat_fc, on=["unique_id", "ds"], how="left")
n_missing = df_stat_eval["pred"].isna().sum()
if n_missing:
    logger.warning(f"  {n_missing:,} statistical predictions missing -> filled with 0")
df_stat_eval["pred"] = df_stat_eval["pred"].fillna(0).clip(lower=0)

stat_m = metrics_all_and_nz(df_stat_eval["actual"], df_stat_eval["pred"], df_stat_eval["weight"])
logger.info(f"Stat  RMSLE_all={stat_m['RMSLE_all']}  RMSLE_nz={stat_m['RMSLE_nz']}")

10:53:51  LGBM  RMSLE_all=0.3686  RMSLE_nz=0.5313
10:53:53  Stat  RMSLE_all=0.6873  RMSLE_nz=0.6432


**Results**

In [19]:
#### results ######################

W = 118
print("\n" + "=" * W)
print("LightGBM and Statistical Benchmarks — case study 2 - Vallés-Pérez et al. (2022)")
print(f"Corporación Favorita | Daily | Period 1: {config['period1_start']} to "
      f"{config['period1_end']} | h={FORECAST_HORIZON} days")
print(f"Series: {n_series:,}  Test rows: {lgbm_m['n_all']:,}  "
      f"Non-zero: {lgbm_m['n_nz']:,} ({100*lgbm_m['n_nz']/lgbm_m['n_all']:.1f}%)")
print("=" * W)

print(f"\nDemand classification (ADI threshold = {ADI_THRESH}):")
for cat, n in df_sbc["category"].value_counts().items():
    mdl = {"smooth": "AutoETS(s=7)", "non-smooth": "CrostonSBA"}[cat]
    print(f"  {cat:<12}: {n:>7,}  ({n/len(df_sbc)*100:5.1f}%)  -> {mdl}")

print(f"\n{'Model':<42} {'RMSLE_all':>10} {'RMSWLE_all':>11} {'MALE_all':>9} "
      f"{'RMSLE_nz':>9} {'RMSWLE_nz':>10} {'MALE_nz':>8}")
print("-" * W)
print(f"{'Paper: Seq2Seq trimmed (mean+-std, 5 runs)':<42} "
      f"{'0.5381':>10} {'0.5377':>11} {'0.3442':>9} {'-':>9} {'-':>10} {'-':>8}")
print(f"{'Ours: Seq2Seq trimmed (seed=655321)':<42} "
      f"{'0.5374':>10} {'0.5370':>11} {'0.3431':>9} {'-':>9} {'-':>10} {'-':>8}")

for label, m in [("Statistical (smooth/non-smooth)", stat_m),
                 ("ML: LightGBM (direct multi-step)", lgbm_m)]:
    print(f"{label:<42} {m['RMSLE_all']:>10.4f} {m['RMSWLE_all']:>11.4f} "
          f"{m['MALE_all']:>9.4f} {m['RMSLE_nz']:>9.4f} {m['RMSWLE_nz']:>10.4f} "
          f"{m['MALE_nz']:>8.4f}")

print(f"\nNotes:")
print(f"  RMSLE  : sqrt(mean((log1p(actual)-log1p(pred))^2)) — unweighted")
print(f"  RMSWLE : perishable weight=1.25, non-perishable=1.0 — the paper's NWRMSLE")
print(f"  MALE   : mean absolute log error")
print(f"  _all   : all {lgbm_m['n_all']:,} test rows, including zero-actual rows")
print(f"  _nz    : the {lgbm_m['n_nz']:,} rows with non-zero actual sales")
print(f"  All-rows metrics favour models that predict near-zero for sparse series;")
print(f"  non-zero metrics are the like-for-like comparison against the seq2seq figures.")
print(f"  LGBM   : direct multi-step, test features fixed at the training cutoff.")
print(f"           Weighted RMSE on log1p target matches the seq2seq loss exactly.")
print(f"  Stat   : univariate — no promotion, holiday, oil or transaction features.")
print(f"  Series : 221,184 from the paper's own cartesian construction, zero-filled.")
print("=" * W)

pd.DataFrame([
    {"model": "Statistical (smooth/non-smooth)", **stat_m},
    {"model": "LightGBM (direct multi-step)",    **lgbm_m},
]).to_csv(os.path.join(RESULTS_DIR, "metrics.csv"), index=False)
print(f"\nOutputs: {RESULTS_DIR}")


LightGBM and Statistical Benchmarks — case study 2 - Vallés-Pérez et al. (2022)
Corporación Favorita | Daily | Period 1: 2017-06-29 to 2017-07-14 | h=16 days
Series: 221,184  Test rows: 3,538,944  Non-zero: 1,699,183 (48.0%)

Demand classification (ADI threshold = 1.32):
  non-smooth  : 185,953  ( 84.1%)  -> CrostonSBA
  smooth      :  35,231  ( 15.9%)  -> AutoETS(s=7)

Model                                       RMSLE_all  RMSWLE_all  MALE_all  RMSLE_nz  RMSWLE_nz  MALE_nz
----------------------------------------------------------------------------------------------------------------------
Paper: Seq2Seq trimmed (mean+-std, 5 runs)     0.5381      0.5377    0.3442         -          -        -
Ours: Seq2Seq trimmed (seed=655321)            0.5374      0.5370    0.3431         -          -        -
Statistical (smooth/non-smooth)                0.6873      0.6856    0.4407    0.6432     0.6415   0.4832
ML: LightGBM (direct multi-step)               0.3686      0.3688    0.2010    0.53